# 13. Regresion diagnostica de determinantes

Este notebook abre la primera etapa de regresion y diagnostico para la base de determinantes. La meta es revisar evidencia antes de aprobar cambios: no ejecuta seleccion automatica, no adopta una regresion reducida y no ajusta una regresion por componentes.

La ejecucion predeterminada es 2024. Para revisar otro anio valido, cambiar unicamente `ANIO_ANALISIS` en la siguiente celda y volver a ejecutar todo el notebook.

In [ ]:
ANIO_ANALISIS = 2024
ANIOS_VALIDOS = (2018, 2020, 2022, 2024)
EDAD_MINIMA = 18
TARGET = "ingreso_persona_laboral_negocio_tri"
CRITERIO_STEPWISE = None
EJECUTAR_SELECCION = False

## Configuracion efectiva

Las variables candidatas usan los nombres originales documentados. `variables_seleccionadas` se define como copia exacta de `variables_candidatas`; cualquier reduccion debe aprobarse despues de revisar los diagnosticos.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis.regresion_diagnostico import (
    ESPECIFICACION,
    EXCLUSIONES_TECNICAS,
    VARIABLES_CANDIDATAS,
    VARIABLES_PENDIENTES,
    build_effective_config,
    run_diagnostics,
)

if ANIO_ANALISIS not in ANIOS_VALIDOS:
    raise ValueError(f"ANIO_ANALISIS debe estar en {ANIOS_VALIDOS}; recibido {ANIO_ANALISIS}.")
if CRITERIO_STEPWISE is not None or EJECUTAR_SELECCION:
    raise ValueError("Esta primera ejecucion no activa stepwise ni seleccion automatica.")

variables_candidatas = VARIABLES_CANDIDATAS.copy()
variables_seleccionadas = variables_candidatas.copy()

configuracion = build_effective_config(
    ANIO_ANALISIS,
    ANIOS_VALIDOS,
    EDAD_MINIMA,
    TARGET,
    CRITERIO_STEPWISE,
    EJECUTAR_SELECCION,
)
pd.Series(configuracion, name="valor").to_frame()

In [ ]:
pd.DataFrame(
    {
        "variable": variables_candidatas,
        "seleccionada_inicialmente": [v in variables_seleccionadas for v in variables_candidatas],
    }
)

In [ ]:
pd.DataFrame(
    [{"variable": variable, "motivo": motivo} for variable, motivo in VARIABLES_PENDIENTES.items()]
)

In [ ]:
pd.DataFrame(EXCLUSIONES_TECNICAS)

## Ejecucion diagnostica

La funcion reutilizable genera particion por hogares, OLS nominal, OLS logaritmico exploratorio, VIF/GVIF, PCA exploratorio y arbol de decision diagnostico. La particion se guarda localmente fuera de Git; las tablas agregadas y figuras se guardan por anio/especificacion.

In [ ]:
resultado = run_diagnostics(
    project_root=PROJECT_ROOT,
    anio_analisis=ANIO_ANALISIS,
    anios_validos=ANIOS_VALIDOS,
    edad_minima=EDAD_MINIMA,
    target=TARGET,
    criterio_stepwise=CRITERIO_STEPWISE,
    ejecutar_seleccion=EJECUTAR_SELECCION,
    variables_candidatas=variables_candidatas,
    variables_seleccionadas=variables_seleccionadas,
)

paths = resultado["paths"]
print(f"Tablas: {paths.table_dir}")
print(f"Figuras: {paths.figure_dir}")
print(f"Reporte: {paths.report_path}")

In [ ]:
def leer_tabla(nombre):
    return pd.read_csv(paths.table_dir / nombre)

def mostrar_figura(nombre):
    display(Image(filename=str(paths.figure_dir / nombre)))

## Entorno

In [ ]:
leer_tabla("dependencias_entorno.csv")

## Compatibilidad, reproduccion y particion

In [ ]:
leer_tabla("compatibilidad_anual_previa.csv")

In [ ]:
leer_tabla("reproduccion_universo_2024.csv")

In [ ]:
leer_tabla("particion_resumen.csv")

In [ ]:
leer_tabla("particion_validaciones.csv")

In [ ]:
leer_tabla("categorias_no_vistas_por_particion.csv")

## OLS nominal y log exploratorio

In [ ]:
leer_tabla("regresion_resumen_ajuste.csv")

In [ ]:
leer_tabla("regresion_metricas.csv")

In [ ]:
mostrar_figura("regresion_residuos_vs_ajustados.png")
mostrar_figura("regresion_residuos_histograma.png")

## VIF/GVIF y dependencias exactas

In [ ]:
leer_tabla("dependencias_matriz_resumen.csv")

In [ ]:
leer_tabla("vif_columnas_entrenamiento.csv").head(15)

In [ ]:
leer_tabla("gvif_bloques_entrenamiento.csv")

In [ ]:
leer_tabla("verificacion_vif_gvif_ejemplos.csv")

## PCA exploratorio

In [ ]:
leer_tabla("pca_varianza.csv").head(20)

In [ ]:
mostrar_figura("pca_scree_varianza.png")
mostrar_figura("pca_varianza_acumulada.png")
mostrar_figura("pca_cargas_heatmap_pc1_pc8_top40.png")

## Arbol diagnostico

In [ ]:
leer_tabla("arbol_metricas.csv")

In [ ]:
leer_tabla("arbol_importancia_impureza_variables.csv")

In [ ]:
leer_tabla("arbol_importancia_permutacion_variables.csv")

## Resumen para revision

La siguiente tabla ordena evidencia diagnostica por variable y deja cada accion como propuesta pendiente. No aplica eliminaciones.

In [ ]:
leer_tabla("tabla_revision_variables.csv")

## Decisiones pendientes

- Definir escala principal del target: ingreso nominal o log ingreso.
- Definir criterio de seleccion posterior si se usa stepwise: AIC, BIC o R2 ajustado.
- Revisar variables laborales con alta dependencia antes de eliminar algo.
- Resolver `segsoc_desc` 2020/2022 antes de comparaciones anuales.
- Definir si `est_socio_desc` y variables de jefatura entran como contexto familiar.
- Definir, en una etapa posterior, si se usara PCA/PCR y cuantos componentes.